# AlphaTrain Pillar 2P: Pure Self-Play (Drop Expert Data)

First pure self-play training. No expert data — the network learns
entirely from distilling its own 1,600-sim search wisdom.

- **1,595 games** at 1,600 sims from 2N model (mean 2,625, 1.97M states)
- **15 epochs** (all data is fresh — no more 1-epoch saturation)
- **Survival hybrid target** (r=1+pts/10, gamma=0.95, 128 bins, max_score=30)
- Warm start from 2N (preserves tactical DNA)

**Upload to Drive (`MyDrive/alphatrain/`):**
1. `colorlines_selfplay_train.tar.gz` — code
2. `selfplay_v4_surv_ms30.pt.gz` — pure self-play data
3. `pillar2n_best.pt` — base model

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, time

DRIVE = '/content/drive/MyDrive/alphatrain'

# Extract code (always fresh)
!cp {DRIVE}/colorlines_selfplay_train.tar.gz /content/
!cd /content && tar xzf colorlines_selfplay_train.tar.gz

# Copy model
os.makedirs('/content/alphatrain/data', exist_ok=True)
print('Copying model...')
shutil.copy(f'{DRIVE}/pillar2n_best.pt', '/content/alphatrain/data/model.pt')
print(f'Model: {os.path.getsize("/content/alphatrain/data/model.pt")/1e6:.0f} MB')

# Decompress self-play v4 data (pure self-play — NO expert data)
print('Decompressing self-play v4 data...')
t0 = time.time()
!gzip -dc {DRIVE}/selfplay_v4_surv_ms30.pt.gz > /content/alphatrain/data/selfplay.pt
print(f'Self-play: {os.path.getsize("/content/alphatrain/data/selfplay.pt")/1e9:.1f} GB ({time.time()-t0:.0f}s)')

# VERIFY all files exist
for f in ['model.pt', 'selfplay.pt']:
    path = f'/content/alphatrain/data/{f}'
    assert os.path.exists(path), f'MISSING: {path}'
    print(f'  OK: {f} ({os.path.getsize(path)/1e6:.0f} MB)')

!pip install -q numpy numba scipy pytest

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {g.total_memory / 1e9:.1f} GB')

d = torch.load('/content/alphatrain/data/selfplay.pt', weights_only=True, map_location='cpu')
print(f'\nSelf-play: {d["boards"].shape[0]:,} states, '
      f'max_score={d["max_score"]}, bins={d["num_value_bins"]}')
del d

!cd /content && python -m pytest alphatrain/tests/test_model.py -v --tb=short 2>&1 | tail -5

In [ ]:
# Pillar 2P: PURE SELF-PLAY — no expert data
# 1,595 games at 1600 sims, 15 epochs (all fresh data)
# Warm start from 2N
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train \
    --tensor-file alphatrain/data/selfplay.pt \
    --gpu-data --amp --compile \
    --value-bins 128 --value-channels 32 --value-hidden 512 --value-dropout 0.3 \
    --val-weight 0.01 --rank-weight 1.0 \
    --endgame-fraction 0.3 --endgame-threshold 100 \
    --adversarial-ranking \
    --resume alphatrain/data/model.pt --warm-start \
    --epochs 15 --batch-size 8192 --lr 1e-4 --warmup-epochs 2 \
    --copy-to /content/drive/MyDrive/alphatrain/pillar2p_best.pt \
    --save-dir /content/checkpoints/pillar2p

In [ ]:
# Copy final model to Drive
import shutil, os
DRIVE = '/content/drive/MyDrive/alphatrain'
for f in ['best.pt', 'latest.pt']:
    src = f'/content/checkpoints/pillar2p/{f}'
    dst = f'{DRIVE}/pillar2p_{f}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'Saved {dst} ({os.path.getsize(dst)/1e6:.0f} MB)')